<a href="https://colab.research.google.com/github/NanThawe/Solar-Filament-Segmentation/blob/main/solar_filament_instance_segmentation_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from pathlib import Path
from dataclasses import dataclass
import gc

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

In [2]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

FULL_RES = 2048
WORK_RES = 1024
BASE_WIDTH = 24

# Final calibrated V5-A parameters
LOCAL_WEIGHT = 0.60
GLOBAL_WEIGHT = 0.40

LOW_THRESHOLD = 0.40
HIGH_THRESHOLD = 0.75
CLOSE_RADIUS = 1
MIN_AREA = 480

In [3]:
DATA_DIR = Path(
    "/content/drive/MyDrive/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
)

TEST_IMAGE_DIR = (
    DATA_DIR
    / "test"
    / "test_images"
)

# Replace with the Kaggle Dataset
# containing your six .pt files
MODEL_DIR = Path(
    "/content/drive/MyDrive/solar_filament_v4c"
)

OUTPUT_PATH = Path(
    "/content/submission.csv"
)

In [4]:
def fit_solar_disk(image):

    blurred = cv2.GaussianBlur(
        image,
        (0, 0),
        3
    )

    _, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY
        + cv2.THRESH_OTSU
    )

    n, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            (binary > 0).astype(
                np.uint8
            )
        )
    )

    if n < 2:

        h, w = image.shape

        return (
            w / 2,
            h / 2,
            min(h, w) * 0.45
        )

    component = (
        1
        + np.argmax(
            stats[
                1:,
                cv2.CC_STAT_AREA
            ]
        )
    )

    disk = (
        labels
        == component
    ).astype(np.uint8)

    contours, _ = cv2.findContours(
        disk,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    contour = max(
        contours,
        key=cv2.contourArea
    )

    (cx, cy), radius = (
        cv2.minEnclosingCircle(
            contour
        )
    )

    return (
        float(cx),
        float(cy),
        float(radius)
    )

In [5]:
def make_radius_map(
    shape,
    cx,
    cy,
    radius
):

    yy, xx = np.mgrid[
        :shape[0],
        :shape[1]
    ]

    r = np.hypot(
        xx - cx,
        yy - cy
    )

    return (
        r / radius
    ).astype(
        np.float32
    )

In [6]:
def radial_profile(
    image,
    radius_map,
    disk_mask,
    bins=100
):

    edges = np.linspace(
        0,
        1,
        bins + 1
    )

    indices = np.clip(
        np.digitize(
            radius_map[
                disk_mask
            ],
            edges
        ) - 1,
        0,
        bins - 1
    )

    values = image[
        disk_mask
    ].astype(np.float32)

    profile = np.zeros(
        bins,
        dtype=np.float32
    )

    centres = (
        edges[:-1]
        + edges[1:]
    ) / 2

    for i in range(bins):

        x = values[
            indices == i
        ]

        if len(x):
            profile[i] = np.median(x)
        else:
            profile[i] = np.nan

    good = ~np.isnan(
        profile
    )

    profile = np.interp(
        centres,
        centres[good],
        profile[good]
    )

    return np.interp(
        radius_map,
        centres,
        profile
    ).astype(np.float32)

In [7]:
def normalized_blur(
    image,
    support,
    sigma
):

    numerator = (
        cv2.GaussianBlur(
            image * support,
            (0, 0),
            sigma
        )
    )

    denominator = (
        cv2.GaussianBlur(
            support,
            (0, 0),
            sigma
        )
    )

    return (
        numerator
        / np.maximum(
            denominator,
            1e-3
        )
    )

In [8]:
def gaussian_kernels(
    sigma
):

    radius = int(
        np.ceil(
            4 * sigma
        )
    )

    x = np.arange(
        -radius,
        radius + 1,
        dtype=np.float32
    )

    g = np.exp(
        -(x ** 2)
        / (2 * sigma ** 2)
    )

    g /= g.sum()

    g1 = (
        -x
        / sigma ** 2
        * g
    )

    g2 = (
        (
            x ** 2
            / sigma ** 4
        )
        -
        (
            1
            / sigma ** 2
        )
    ) * g

    return g, g1, g2

In [9]:
def dark_ridge_response(
    image,
    sigmas
):

    ridge = np.zeros_like(
        image,
        dtype=np.float32
    )

    for sigma in sigmas:

        g, g1, g2 = (
            gaussian_kernels(
                sigma
            )
        )

        ixx = cv2.sepFilter2D(
            image,
            cv2.CV_32F,
            g2,
            g
        )

        iyy = cv2.sepFilter2D(
            image,
            cv2.CV_32F,
            g,
            g2
        )

        ixy = cv2.sepFilter2D(
            image,
            cv2.CV_32F,
            g1,
            g1
        )

        trace = (
            0.5
            * (ixx + iyy)
        )

        discriminant = np.sqrt(
            np.maximum(
                0.25
                * (ixx - iyy) ** 2
                + ixy ** 2,
                0
            )
        )

        lambda_max = (
            sigma ** 2
            * (
                trace
                + discriminant
            )
        )

        ridge = np.maximum(
            ridge,
            np.maximum(
                lambda_max,
                0
            )
        )

    return ridge

In [10]:
def build_input_channels(
    image_full
):

    W = V5CFG.work_res

    image = cv2.resize(
        image_full,
        (W, W),
        interpolation=cv2.INTER_AREA
    )

    cx, cy, radius = (
        fit_solar_disk(
            image
        )
    )

    radius_map = (
        make_radius_map(
            image.shape,
            cx,
            cy,
            radius
        )
    )

    disk = (
        radius_map
        < V5CFG.disk_fraction
    )

    profile = radial_profile(
        image,
        radius_map,
        disk
    )

    flat = (
        image.astype(np.float32)
        /
        np.maximum(
            profile,
            1
        )
    )

    flat[
        ~disk
    ] = 1

    support = disk.astype(
        np.float32
    )

    local_mean = normalized_blur(
        flat,
        support,
        V5CFG.local_mean_sigma
    )

    difference = (
        flat
        - local_mean
    ) * support

    local_std = np.sqrt(
        np.maximum(
            normalized_blur(
                difference ** 2,
                support,
                V5CFG.local_std_sigma
            ),
            1e-8
        )
    )

    zscore = (
        difference
        / np.maximum(
            local_std,
            0.02
        )
    )

    ridge = dark_ridge_response(
        flat,
        V5CFG.ridge_sigmas
    )

    ridge *= support

    scale = (
        np.median(
            ridge[disk]
        )
        + 1e-6
    )

    ridge = np.log1p(
        ridge
        / (4 * scale)
    )

    channel_0 = np.clip(
        (flat - 1) / 0.25,
        -3,
        3
    )

    channel_1 = np.clip(
        zscore,
        -4,
        4
    )

    channel_2 = np.clip(
        ridge,
        0,
        4
    )

    channels = np.stack(
        [
            channel_0,
            channel_1,
            channel_2
        ]
    )

    channels *= (
        disk[None]
    )

    return (
        channels.astype(
            np.float16
        ),
        disk,
        (cx, cy, radius)
    )

Model Architecture

In [11]:
# MulitRes Model
class ConvGNAct(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            group_norm(
                out_channels
            ),

            nn.GELU()
        )

    def forward(
        self,
        x
    ):

        return self.block(x)

In [12]:
class MultiResBlock(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        c1 = max(
            out_channels // 6,
            1
        )

        c2 = max(
            out_channels // 3,
            1
        )

        c3 = (
            out_channels
            - c1
            - c2
        )

        self.path1 = ConvGNAct(
            in_channels,
            c1
        )

        self.path2 = ConvGNAct(
            c1,
            c2
        )

        self.path3 = ConvGNAct(
            c2,
            c3
        )

        self.residual = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.norm = group_norm(
            out_channels
        )

    def forward(
        self,
        x
    ):

        p1 = self.path1(x)

        p2 = self.path2(
            p1
        )

        p3 = self.path3(
            p2
        )

        multi = torch.cat(
            [
                p1,
                p2,
                p3
            ],
            dim=1
        )

        residual = (
            self.residual(x)
        )

        return F.gelu(
            self.norm(
                multi
                + residual
            )
        )

In [13]:
class AttentionGate(
    nn.Module
):

    def __init__(
        self,
        skip_channels,
        gate_channels
    ):

        super().__init__()

        intermediate = max(
            skip_channels // 2,
            8
        )

        self.skip_projection = (
            nn.Conv2d(
                skip_channels,
                intermediate,
                1
            )
        )

        self.gate_projection = (
            nn.Conv2d(
                gate_channels,
                intermediate,
                1
            )
        )

        self.attention = (
            nn.Conv2d(
                intermediate,
                1,
                1
            )
        )

    def forward(
        self,
        skip,
        gate
    ):

        attention = (
            self.skip_projection(
                skip
            )
            +
            self.gate_projection(
                gate
            )
        )

        attention = F.gelu(
            attention
        )

        attention = torch.sigmoid(
            self.attention(
                attention
            )
        )

        return (
            skip
            * attention
        )

In [14]:
class MultiResAttentionUNet(
    nn.Module
):

    def __init__(
        self,
        in_channels=3,
        base=24
    ):

        super().__init__()

        widths = [
            base,
            base * 2,
            base * 4,
            base * 8,
            base * 16
        ]

        self.enc1 = MultiResBlock(
            in_channels,
            widths[0]
        )

        self.enc2 = MultiResBlock(
            widths[0],
            widths[1]
        )

        self.enc3 = MultiResBlock(
            widths[1],
            widths[2]
        )

        self.enc4 = MultiResBlock(
            widths[2],
            widths[3]
        )

        self.bridge = MultiResBlock(
            widths[3],
            widths[4]
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.up4 = nn.Conv2d(
            widths[4],
            widths[3],
            1
        )

        self.att4 = AttentionGate(
            widths[3],
            widths[3]
        )

        self.dec4 = MultiResBlock(
            widths[3] * 2,
            widths[3]
        )

        self.up3 = nn.Conv2d(
            widths[3],
            widths[2],
            1
        )

        self.att3 = AttentionGate(
            widths[2],
            widths[2]
        )

        self.dec3 = MultiResBlock(
            widths[2] * 2,
            widths[2]
        )

        self.up2 = nn.Conv2d(
            widths[2],
            widths[1],
            1
        )

        self.att2 = AttentionGate(
            widths[1],
            widths[1]
        )

        self.dec2 = MultiResBlock(
            widths[1] * 2,
            widths[1]
        )

        self.up1 = nn.Conv2d(
            widths[1],
            widths[0],
            1
        )

        self.att1 = AttentionGate(
            widths[0],
            widths[0]
        )

        self.dec1 = MultiResBlock(
            widths[0] * 2,
            widths[0]
        )

        # Main semantic head
        self.mask_head = nn.Conv2d(
            widths[0],
            1,
            1
        )

        # Auxiliary topology/spine head
        self.spine_head = nn.Conv2d(
            widths[0],
            1,
            1
        )

    def upsample(
        self,
        x,
        target
    ):

        return F.interpolate(
            x,
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    def forward(
        self,
        x
    ):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        bridge = self.bridge(
            self.pool(e4)
        )

        d4 = self.upsample(
            bridge,
            e4
        )

        d4 = self.up4(d4)

        s4 = self.att4(
            e4,
            d4
        )

        d4 = self.dec4(
            torch.cat(
                [d4, s4],
                dim=1
            )
        )

        d3 = self.upsample(
            d4,
            e3
        )

        d3 = self.up3(d3)

        s3 = self.att3(
            e3,
            d3
        )

        d3 = self.dec3(
            torch.cat(
                [d3, s3],
                dim=1
            )
        )

        d2 = self.upsample(
            d3,
            e2
        )

        d2 = self.up2(d2)

        s2 = self.att2(
            e2,
            d2
        )

        d2 = self.dec2(
            torch.cat(
                [d2, s2],
                dim=1
            )
        )

        d1 = self.upsample(
            d2,
            e1
        )

        d1 = self.up1(d1)

        s1 = self.att1(
            e1,
            d1
        )

        d1 = self.dec1(
            torch.cat(
                [d1, s1],
                dim=1
            )
        )

        return {
            "mask":
                self.mask_head(d1),

            "spine":
                self.spine_head(d1)
        }

Physics Preprocessing

In [15]:
class V5Config:
    full_res = 2048
    crop_size = 512
    tile_stride = 384
    work_res: int = 1024

    base_width = 24

    local_mean_sigma = 32.0
    local_std_sigma = 64.0

    ridge_sigmas = (
        1.5,
        3.0,
        6.0,
        12.0,
    )

    ridge_pad = 64

    # Set to 2 if GPU memory is tight
    tile_batch_size = 4

# Physics channels
    disk_fraction: float = 0.985
    local_mean_sigma: float = 16.0
    local_std_sigma: float = 32.0
    ridge_sigmas: tuple = (1.5, 3.0, 6.0)


V5CFG = V5Config()

In [16]:
def prepare_v5a_raw(
    raw
):

    if raw.shape != (
        V5CFG.full_res,
        V5CFG.full_res
    ):

        raw = cv2.resize(
            raw,
            (
                V5CFG.full_res,
                V5CFG.full_res
            ),
            interpolation=cv2.INTER_AREA
        )

    return raw

In [17]:
def build_v5a_native_base(
    raw
):

    raw = prepare_v5a_raw(
        raw
    )

    cx, cy, radius = (
        fit_solar_disk(
            raw
        )
    )

    radius_map = (
        make_radius_map(
            raw.shape,
            cx,
            cy,
            radius
        )
    )

    disk = (
        radius_map
        < V5CFG.disk_fraction
    )

    profile = radial_profile(
        raw,
        radius_map,
        disk,
        bins=160
    )

    flat = (
        raw.astype(
            np.float32
        )
        /
        np.maximum(
            profile,
            1.0
        )
    )

    flat[
        ~disk
    ] = 1.0


    support = disk.astype(
        np.float32
    )


    # --------------------------------
    # Native local contrast
    # --------------------------------

    local_mean = (
        normalized_blur(
            flat,
            support,
            V5CFG.local_mean_sigma
        )
    )

    difference = (
        flat
        - local_mean
    ) * support

    local_std = np.sqrt(
        np.maximum(

            normalized_blur(
                difference ** 2,
                support,
                V5CFG.local_std_sigma
            ),

            1e-8
        )
    )

    zscore = (
        difference
        /
        np.maximum(
            local_std,
            0.02
        )
    )


    channel_0 = np.clip(
        (flat - 1.0) / 0.25,
        -3,
        3
    )

    channel_1 = np.clip(
        zscore,
        -4,
        4
    )


    channel_0 *= support
    channel_1 *= support


    return {

        "flat":
            flat.astype(
                np.float32
            ),

        "channel_0":
            channel_0.astype(
                np.float32
            ),

        "channel_1":
            channel_1.astype(
                np.float32
            ),

        "disk":
            disk
    }


def build_v5a_tile(
    base,
    global_mask_full,
    global_spine_full,
    y0,
    x0
):

    C = V5CFG.crop_size
    P = V5CFG.ridge_pad

    H, W = base[
        "flat"
    ].shape


    # Padded region for ridge detection
    ys = max(
        0,
        y0 - P
    )

    xs = max(
        0,
        x0 - P
    )

    ye = min(
        H,
        y0 + C + P
    )

    xe = min(
        W,
        x0 + C + P
    )


    flat_sub = (
        base[
            "flat"
        ][
            ys:ye,
            xs:xe
        ]
    )

    disk_sub = (
        base[
            "disk"
        ][
            ys:ye,
            xs:xe
        ]
    )


    ridge_sub = (
        dark_ridge_response(
            flat_sub,
            V5CFG.ridge_sigmas
        )
    )

    ridge_sub *= (
        disk_sub.astype(
            np.float32
        )
    )


    ridge_values = (
        ridge_sub[
            disk_sub
        ]
    )

    if len(
        ridge_values
    ) > 0:

        ridge_scale = float(
            np.median(
                ridge_values
            )
        )

    else:

        ridge_scale = 1.0


    if ridge_scale < 1e-6:

        ridge_scale = (
            float(
                np.percentile(
                    ridge_values,
                    75
                )
            )
            + 1e-6

            if len(
                ridge_values
            ) > 0

            else 1.0
        )


    ridge_sub = np.log1p(
        ridge_sub
        /
        (
            4.0
            * ridge_scale
            + 1e-6
        )
    )

    ridge_sub = np.clip(
        ridge_sub,
        0,
        4
    )


    ry0 = (
        y0 - ys
    )

    rx0 = (
        x0 - xs
    )


    channel_2 = (
        ridge_sub[
            ry0:ry0+C,
            rx0:rx0+C
        ]
    )


    channel_0 = (
        base[
            "channel_0"
        ][
            y0:y0+C,
            x0:x0+C
        ]
    )

    channel_1 = (
        base[
            "channel_1"
        ][
            y0:y0+C,
            x0:x0+C
        ]
    )

    channel_3 = (
        global_mask_full[
            y0:y0+C,
            x0:x0+C
        ]
    )

    channel_4 = (
        global_spine_full[
            y0:y0+C,
            x0:x0+C
        ]
    )


    channels = np.stack(
        [
            channel_0,
            channel_1,
            channel_2,
            channel_3,
            channel_4
        ],
        axis=0
    ).astype(
        np.float32
    )


    valid = (
        base[
            "disk"
        ][
            y0:y0+C,
            x0:x0+C
        ]
        .astype(
            np.float32
        )
    )


    channels *= (
        valid[None]
    )


    return (
        channels,
        valid
    )

In [18]:
def v5a_tile_positions():

    C = V5CFG.crop_size
    S = V5CFG.tile_stride
    W = V5CFG.full_res


    positions = list(
        range(
            0,
            W - C + 1,
            S
        )
    )


    if (
        positions[-1]
        != W - C
    ):

        positions.append(
            W - C
        )


    return positions


def v5a_blend_window():

    w = np.hanning(
        V5CFG.crop_size
    ).astype(
        np.float32
    )

    # Prevent zero weight at
    # outer image boundaries
    w = np.maximum(
        w,
        0.05
    )

    window = np.outer(
        w,
        w
    )

    return window.astype(
        np.float32
    )

Instance Post-processing

In [19]:
def hysteresis_instances(
    probability,
    valid,

    low_threshold,
    high_threshold,

    close_radius=3,
    min_area=30
):

    low_mask = (
        (probability >= low_threshold)
        & valid
    )

    n, labels = (
        cv2.connectedComponents(
            low_mask.astype(
                np.uint8
            ),
            connectivity=8
        )
    )

    if n <= 1:

        return np.zeros(
            probability.shape,
            dtype=np.int32
        )

    high_mask = (
        (probability >= high_threshold)
        & valid
    )

    seed_ids = np.unique(
        labels[
            high_mask
        ]
    )

    seed_ids = seed_ids[
        seed_ids > 0
    ]

    mask = np.isin(
        labels,
        seed_ids
    )

    if close_radius > 0:

        kernel = (
            cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,

                (
                    2 * close_radius + 1,
                    2 * close_radius + 1
                )
            )
        )

        mask = cv2.morphologyEx(
            mask.astype(
                np.uint8
            ),

            cv2.MORPH_CLOSE,

            kernel
        ).astype(bool)

    mask &= valid

    mask = ndi.binary_fill_holes(
        mask
    )

    n, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask.astype(
                np.uint8
            ),
            connectivity=8
        )
    )

    output = np.zeros(
        probability.shape,
        dtype=np.int32
    )

    next_id = 1

    for component_id in range(
        1,
        n
    ):

        area = stats[
            component_id,
            cv2.CC_STAT_AREA
        ]

        if area < min_area:
            continue

        output[
            labels
            == component_id
        ] = next_id

        next_id += 1

    return output

Load six pretrained models

In [20]:
def group_norm(
    channels
):

    possible = [
        g
        for g in range(
            1,
            min(
                8,
                channels
            ) + 1
        )
        if channels % g == 0
    ]

    groups = max(
        possible
    )

    return nn.GroupNorm(
        groups,
        channels
    )

def load_model(path, in_channels):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model = MultiResAttentionUNet(
        in_channels=in_channels,
        base=BASE_WIDTH
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    model.eval()

    return model


GLOBAL_MODELS = {
    fold: load_model(
        MODEL_DIR / f"v4d_fold{fold}.pt",
        in_channels=3
    )
    for fold in range(3)
}


LOCAL_MODELS = {
    fold: load_model(
        MODEL_DIR / f"v5a_local_fold{fold}.pt",
        in_channels=5
    )
    for fold in range(3)
}


print("Loaded 3 V4-D + 3 V5-A models")

Loaded 3 V4-D + 3 V5-A models


3-fold Inference

In [21]:
@torch.inference_mode()
def predict_global_heads_from_channels(
    model,
    channels
):

    x = torch.from_numpy(
        channels.astype(
            np.float32
        )
    )[None].to(
        DEVICE
    )

    model.eval()

    with torch.autocast(
        device_type=DEVICE.type,
        enabled=(
            DEVICE.type == "cuda"
        )
    ):

        output = model(
            x
        )


    mask_probability = (
        torch.sigmoid(
            output["mask"]
        )[0, 0]
        .float()
        .cpu()
        .numpy()
    )


    spine_probability = (
        torch.sigmoid(
            output["spine"]
        )[0, 0]
        .float()
        .cpu()
        .numpy()
    )


    del x
    del output


    return (
        mask_probability,
        spine_probability
    )


# ------------------------------------------------------------
# Precompute native image features once
# ------------------------------------------------------------

def precompute_v5a_static_tiles(
    base
):

    positions = (
        v5a_tile_positions()
    )

    coordinates = [

        (y0, x0)

        for y0 in positions

        for x0 in positions
    ]


    zero_prior = np.zeros(
        (
            V5CFG.full_res,
            V5CFG.full_res
        ),
        dtype=np.float32
    )


    static_tiles = []


    for y0, x0 in coordinates:

        tile, _ = (
            build_v5a_tile(
                base,
                zero_prior,
                zero_prior,
                y0,
                x0
            )
        )

        # Physics channels only.
        # Store FP16 to reduce RAM.
        static_tiles.append(
            tile[
                :3
            ].astype(
                np.float16
            )
        )


    del zero_prior


    return (
        coordinates,
        static_tiles
    )


# ------------------------------------------------------------
# Native local inference for one fold
# ------------------------------------------------------------

@torch.inference_mode()
def predict_local_from_static_tiles(
    model,
    base,
    coordinates,
    static_tiles,
    global_mask_full,
    global_spine_full
):

    C = V5CFG.crop_size

    probability_sum = np.zeros(
        (
            V5CFG.full_res,
            V5CFG.full_res
        ),
        dtype=np.float32
    )

    weight_sum = np.zeros_like(
        probability_sum
    )


    window = (
        v5a_blend_window()
    )


    batch_size = (
        V5CFG.tile_batch_size
    )


    for start in range(
        0,
        len(coordinates),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(coordinates)
        )


        tiles = []


        for index in range(
            start,
            end
        ):

            y0, x0 = (
                coordinates[
                    index
                ]
            )


            tile = np.empty(
                (
                    5,
                    C,
                    C
                ),
                dtype=np.float32
            )


            tile[:3] = (
                static_tiles[
                    index
                ]
            )


            tile[3] = (
                global_mask_full[
                    y0:y0+C,
                    x0:x0+C
                ]
            )


            tile[4] = (
                global_spine_full[
                    y0:y0+C,
                    x0:x0+C
                ]
            )


            tiles.append(
                tile
            )


        x = torch.from_numpy(
            np.stack(
                tiles
            )
        ).to(
            DEVICE
        )


        model.eval()


        with torch.autocast(
            device_type=DEVICE.type,
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            output = model(
                x
            )


        probabilities = (
            torch.sigmoid(
                output["mask"]
            )[:, 0]
            .float()
            .cpu()
            .numpy()
        )


        for local_index, probability in enumerate(
            probabilities
        ):

            index = (
                start
                + local_index
            )

            y0, x0 = (
                coordinates[
                    index
                ]
            )


            probability_sum[
                y0:y0+C,
                x0:x0+C
            ] += (
                probability
                * window
            )


            weight_sum[
                y0:y0+C,
                x0:x0+C
            ] += window


        del x
        del output
        del probabilities


    probability = (
        probability_sum
        /
        np.maximum(
            weight_sum,
            1e-6
        )
    )


    probability *= (
        base["disk"]
    )


    return probability.astype(
        np.float32
    )


# ------------------------------------------------------------
# Complete 3-fold ensemble for one test image
# ------------------------------------------------------------

@torch.inference_mode()
def infer_v5a_test_image_3fold(
    image_path
):

    raw = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    assert raw is not None, (
        image_path
    )


    raw = (
        prepare_v5a_raw(
            raw
        )
    )


    # --------------------------------
    # Global 1024 physics channels
    # computed once
    # --------------------------------

    global_channels, _, _ = (
        build_input_channels(
            raw
        )
    )


    # --------------------------------
    # Native full-resolution physics
    # computed once
    # --------------------------------

    base = (
        build_v5a_native_base(
            raw
        )
    )


    (
        coordinates,
        static_tiles
    ) = (
        precompute_v5a_static_tiles(
            base
        )
    )


    ensemble_sum = np.zeros(
        (
            V5CFG.full_res,
            V5CFG.full_res
        ),
        dtype=np.float32
    )


    # ================================================
    # FOLD ENSEMBLE
    # ================================================

    for fold in [0, 1, 2]:

        # --------------------------------
        # GLOBAL V4-D MODEL
        # --------------------------------

        global_model = (
            GLOBAL_MODELS[
                fold
            ]
            .to(
                DEVICE
            )
        )


        (
            global_mask_small,
            global_spine_small
        ) = (
            predict_global_heads_from_channels(
                global_model,
                global_channels
            )
        )


        # Move global model back to CPU
        GLOBAL_MODELS[
            fold
        ] = (
            global_model
            .to("cpu")
        )

        del global_model


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        # --------------------------------
        # Upsample global context to 2048
        # --------------------------------

        global_mask_full = cv2.resize(
            global_mask_small,
            (
                V5CFG.full_res,
                V5CFG.full_res
            ),
            interpolation=cv2.INTER_LINEAR
        )


        global_spine_full = cv2.resize(
            global_spine_small,
            (
                V5CFG.full_res,
                V5CFG.full_res
            ),
            interpolation=cv2.INTER_LINEAR
        )


        global_mask_full *= (
            base["disk"]
        )

        global_spine_full *= (
            base["disk"]
        )


        # --------------------------------
        # LOCAL NATIVE V5-A MODEL
        # --------------------------------

        local_model = (
            LOCAL_MODELS[
                fold
            ]
            .to(
                DEVICE
            )
        )


        local_probability = (
            predict_local_from_static_tiles(
                local_model,
                base,
                coordinates,
                static_tiles,
                global_mask_full,
                global_spine_full
            )
        )


        # Move local model back to CPU
        LOCAL_MODELS[
            fold
        ] = (
            local_model
            .to("cpu")
        )

        del local_model


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        # --------------------------------
        # Same calibrated global/local
        # fusion used during OOF
        # --------------------------------

        fold_probability = (
    LOCAL_WEIGHT * local_probability
    + GLOBAL_WEIGHT * global_mask_full
)


        del global_mask_small
        del global_spine_small
        del global_mask_full
        del global_spine_full
        del local_probability
        del fold_probability


    # ================================================
    # AVERAGE THE THREE FOLDS
    # ================================================

    ensemble_probability = (
        ensemble_sum
        / 3.0
    )


    ensemble_probability *= (
        base["disk"]
    )


    # ================================================
    # INSTANCE EXTRACTION ONCE, AFTER ENSEMBLING
    # ================================================

    labels = hysteresis_instances(
    ensemble_probability,
    base["disk"],
    low_threshold=LOW_THRESHOLD,
    high_threshold=HIGH_THRESHOLD,
    close_radius=CLOSE_RADIUS,
    min_area=MIN_AREA,
)


    return (
        ensemble_probability,
        labels
    )

In [22]:
# ============================================================
# V5-A.28 — NATIVE 2048 RLE SUBMISSION ENCODER
# ============================================================

def labels_to_native_submission_rows(
    labels,
    image_id
):

    assert labels.shape == (
        V5CFG.full_res,
        V5CFG.full_res
    )


    rows = []


    component_ids = [

        int(component_id)

        for component_id
        in np.unique(
            labels
        )

        if component_id > 0
    ]


    # Deterministic:
    # largest components first
    component_ids.sort(

        key=lambda component_id:

            int(
                np.sum(
                    labels
                    == component_id
                )
            ),

        reverse=True
    )


    for instance_number, component_id in enumerate(
        component_ids,
        start=1
    ):

        mask = (
            labels
            == component_id
        ).astype(
            np.uint8
        )


        rle = (
            mask_utils.encode(
                np.asfortranarray(
                    mask
                )
            )
        )


        counts = (
            rle["counts"]
        )


        if isinstance(
            counts,
            bytes
        ):

            counts = (
                counts.decode(
                    "utf-8"
                )
            )


        rows.append({

            "filament_id":
                (
                    f"{image_id}_"
                    f"{instance_number}"
                ),

            "segmentation_rle":
                counts
        })


    return rows

Generate Submission

In [23]:
VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
}

test_files = sorted([
    p
    for p in TEST_IMAGE_DIR.iterdir()
    if p.suffix.lower()
    in VALID_EXTENSIONS
])

print(
    "Test images:",
    len(test_files)
)

assert len(test_files) == 180


submission_rows = []


for image_path in tqdm(
    test_files,
    desc="V5-A inference"
):

    _, labels = (
        infer_v5a_test_image_3fold(
            image_path
        )
    )

    submission_rows.extend(
        labels_to_native_submission_rows(
            labels,
            image_path.stem
        )
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


submission = pd.DataFrame(
    submission_rows,
    columns=[
        "filament_id",
        "segmentation_rle"
    ]
)


# Minimal safety checks
assert (
    submission[
        "filament_id"
    ]
    .notna()
    .all()
)

assert (
    submission[
        "segmentation_rle"
    ]
    .notna()
    .all()
)

assert not (
    submission[
        "filament_id"
    ]
    .duplicated()
    .any()
)


submission.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    f"Saved {len(submission)} "
    f"instances to {OUTPUT_PATH}"
)

display(
    submission.head()
)

Test images: 180


V5-A inference:   0%|          | 0/180 [00:00<?, ?it/s]

Saved 0 instances to /content/submission.csv


,filament_id,segmentation_rle
